# Phase 4: Portfolio Construction & Risk Management  
## Notebook 04_02 — CAPM and Performance Attribution

### Research question

How do we use CAPM as the first performance-attribution model, then show why richer factor models such as Fama-French and Carhart often explain away apparent alpha?

This notebook follows:

```text
04_01_multi_asset_return_engine
```

The previous notebook built the return-accounting foundation. This notebook takes portfolio returns and asks:

> Was performance true alpha, market beta, size/value/momentum exposure, quality/profitability exposure, investment exposure, or residual noise?

The notebook deliberately treats CAPM as the historical baseline, not the final model.

It covers:

1. CAPM as the one-factor benchmark;
2. Fama-French 3-factor intuition;
3. Carhart 4-factor intuition;
4. Fama-French 5-factor intuition;
5. synthetic factor return generation;
6. synthetic portfolios with known exposures;
7. excess return construction;
8. OLS factor regressions;
9. alpha, beta, t-statistics, and $R^2$;
10. alpha shrinkage from CAPM to richer models;
11. train/test attribution;
12. factor contribution decomposition;
13. systematic versus residual risk;
14. rolling beta and rolling alpha;
15. benchmark-relative attribution;
16. model limitations and governance checks;
17. saved outputs and manifest.

The key idea:

> CAPM is a useful first lens, but many “alphas” are just missing factor exposures.

## 1. Why CAPM still matters

CAPM is the simplest factor model:

$$
\begin{aligned}
R_{p,t}-R_{f,t} &= \alpha_p \\
&\quad + \beta_p(R_{m,t}-R_{f,t}) \\
&\quad + \epsilon_{p,t}
\end{aligned}
$$

It says a portfolio's excess return should be explained by exposure to the market risk premium.

CAPM is old and incomplete, but it remains useful because:

1. it gives the first estimate of market beta;
2. it gives a first-pass alpha;
3. it separates market exposure from residual return;
4. it provides a baseline to compare richer models.

But CAPM often overstates alpha because it omits other systematic factors.

## 2. From CAPM to richer factor models

### CAPM

$$
R_p-R_f=\alpha+\beta_m MKT+\epsilon
$$

### Fama-French 3-factor model

$$
\begin{aligned}
R_p-R_f &= \alpha \\
&\quad + \beta_m MKT \\
&\quad + \beta_s SMB \\
&\quad + \beta_h HML \\
&\quad + \epsilon
\end{aligned}
$$

where:

- $MKT$: market excess return;
- $SMB$: size factor, small minus big;
- $HML$: value factor, high book-to-market minus low.

### Carhart 4-factor model

$$
\begin{aligned}
R_p-R_f &= \alpha \\
&\quad + \beta_m MKT \\
&\quad + \beta_s SMB \\
&\quad + \beta_h HML \\
&\quad + \beta_{mom} MOM \\
&\quad + \epsilon
\end{aligned}
$$

Carhart adds momentum.

### Fama-French 5-factor model

$$
\begin{aligned}
R_p-R_f &= \alpha \\
&\quad + \beta_m MKT \\
&\quad + \beta_s SMB \\
&\quad + \beta_h HML \\
&\quad + \beta_r RMW \\
&\quad + \beta_c CMA \\
&\quad + \epsilon
\end{aligned}
$$

where:

- $RMW$: profitability, robust minus weak;
- $CMA$: investment, conservative minus aggressive.

In practice, many researchers also examine:

$$
FF5 + MOM
$$

because momentum remains economically important in many attribution workflows.

## 3. Performance attribution

For a factor model:

$$
\begin{aligned}
R_{p,t}-R_{f,t} &= \alpha \\
&\quad + \sum_{j=1}^{K}\beta_j f_{j,t} \\
&\quad + \epsilon_t
\end{aligned}
$$

Average excess return can be decomposed as:

$$
\begin{aligned}
\mathbb E[R_p-R_f] &= \alpha \\
&\quad + \sum_j \beta_j\mathbb E[f_j] \\
&\quad + \mathbb E[\epsilon]
\end{aligned}
$$

In realised data, the residual average is not always exactly zero out of sample.

So out-of-sample attribution should report:

1. alpha contribution;
2. factor contributions;
3. residual contribution;
4. total explained versus unexplained return;
5. residual risk.

This is the bridge from returns to portfolio risk management.

## 4. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass(frozen=True)
class FactorAttributionConfig:
    n_days: int = 1_800
    train_fraction: float = 0.65
    annualisation: int = 252
    rolling_window: int = 252
    initial_capital: float = 1_000_000.0
    seed: int = 42


config = FactorAttributionConfig()
config

## 5. Simulate factor returns

We simulate daily returns for:

| Factor | Meaning |
|---|---|
| `MKT` | market excess return |
| `SMB` | size |
| `HML` | value |
| `MOM` | momentum |
| `RMW` | profitability |
| `CMA` | investment |
| `RF` | risk-free return |

The simulation includes:

- realistic factor correlations;
- different factor volatilities;
- stress regimes;
- a small daily risk-free rate.

This is synthetic and educational.

In [ ]:
def simulate_factor_returns(config: FactorAttributionConfig) -> pd.DataFrame:
    rng = np.random.default_rng(config.seed)
    n = config.n_days
    dates = pd.bdate_range("2017-01-01", periods=n)

    factor_names = ["MKT", "SMB", "HML", "MOM", "RMW", "CMA"]

    annual_means = np.array([0.060, 0.020, 0.025, 0.035, 0.020, 0.015])
    annual_vols = np.array([0.160, 0.100, 0.110, 0.130, 0.080, 0.070])

    daily_means = annual_means / config.annualisation
    daily_vols = annual_vols / np.sqrt(config.annualisation)

    corr = np.array([
        [ 1.00,  0.30, -0.20, -0.15, -0.10, -0.05],
        [ 0.30,  1.00,  0.15, -0.05, -0.05,  0.10],
        [-0.20,  0.15,  1.00, -0.35,  0.25,  0.30],
        [-0.15, -0.05, -0.35,  1.00,  0.10, -0.20],
        [-0.10, -0.05,  0.25,  0.10,  1.00,  0.20],
        [-0.05,  0.10,  0.30, -0.20,  0.20,  1.00],
    ])

    cov = np.outer(daily_vols, daily_vols) * corr

    stress = rng.uniform(size=n) < 0.015
    returns = rng.multivariate_normal(daily_means, cov, size=n)

    # Stress days: market down, momentum often positive, value/profitability mixed.
    stress_shocks = rng.multivariate_normal(
        np.array([-0.035, -0.012, 0.004, 0.018, 0.004, 0.003]),
        cov * 4.0,
        size=stress.sum()
    )
    returns[stress] += stress_shocks

    rf_daily = np.full(n, 0.02 / config.annualisation)
    rf_daily += 0.000005 * rng.standard_normal(n)
    rf_daily = np.clip(rf_daily, -0.00005, 0.00020)

    out = pd.DataFrame(returns, columns=factor_names)
    out.insert(0, "date", dates)
    out["RF"] = rf_daily
    out["stress_day"] = stress

    return out


factors = simulate_factor_returns(config)
factors.head()

In [ ]:
plt.figure(figsize=(12, 6))
for col in ["MKT", "SMB", "HML", "MOM", "RMW", "CMA"]:
    cumulative = (1 + factors[col]).cumprod()
    plt.plot(factors["date"], cumulative, label=col)
plt.title("Synthetic Factor Cumulative Returns")
plt.xlabel("Date")
plt.ylabel("Growth of 1")
plt.legend()
plt.show()

plt.figure(figsize=(10, 6))
plt.imshow(factors[["MKT", "SMB", "HML", "MOM", "RMW", "CMA"]].corr(), vmin=-1, vmax=1)
plt.colorbar(label="Correlation")
plt.xticks(range(6), ["MKT", "SMB", "HML", "MOM", "RMW", "CMA"])
plt.yticks(range(6), ["MKT", "SMB", "HML", "MOM", "RMW", "CMA"])
plt.title("Factor Return Correlation")
plt.tight_layout()
plt.show()

## 6. Simulate portfolios with known exposures

We simulate several portfolios:

| Portfolio | Intended behaviour |
|---|---|
| `Market_Core` | mostly market beta |
| `Small_Value` | loads on SMB and HML |
| `Momentum_Growth` | loads on MKT and MOM |
| `Quality_Defensive` | loads on RMW and low market beta |
| `Conservative_Balanced` | loads on CMA and bonds-like defensiveness |
| `Fake_CAPM_Alpha` | looks like alpha under CAPM because of missing factor exposures |
| `True_Alpha` | has genuine synthetic alpha |
| `Noisy_Manager` | mostly noise and residual risk |

This lets us check whether the models recover known structure.

In [ ]:
def simulate_portfolio_returns(factors: pd.DataFrame, config: FactorAttributionConfig) -> tuple[pd.DataFrame, pd.DataFrame]:
    rng = np.random.default_rng(config.seed + 200)

    exposure_rows = [
        {"portfolio": "Market_Core", "alpha_ann": 0.000, "MKT": 1.00, "SMB": 0.00, "HML": 0.00, "MOM": 0.00, "RMW": 0.00, "CMA": 0.00, "idio_vol_ann": 0.035},
        {"portfolio": "Small_Value", "alpha_ann": 0.000, "MKT": 0.95, "SMB": 0.70, "HML": 0.65, "MOM": -0.10, "RMW": 0.10, "CMA": 0.20, "idio_vol_ann": 0.060},
        {"portfolio": "Momentum_Growth", "alpha_ann": 0.000, "MKT": 1.10, "SMB": -0.15, "HML": -0.35, "MOM": 0.85, "RMW": 0.15, "CMA": -0.10, "idio_vol_ann": 0.065},
        {"portfolio": "Quality_Defensive", "alpha_ann": 0.000, "MKT": 0.65, "SMB": -0.10, "HML": 0.10, "MOM": 0.15, "RMW": 0.75, "CMA": 0.20, "idio_vol_ann": 0.045},
        {"portfolio": "Conservative_Balanced", "alpha_ann": 0.000, "MKT": 0.55, "SMB": -0.05, "HML": 0.20, "MOM": 0.05, "RMW": 0.25, "CMA": 0.70, "idio_vol_ann": 0.040},
        {"portfolio": "Fake_CAPM_Alpha", "alpha_ann": 0.000, "MKT": 0.90, "SMB": 0.45, "HML": 0.55, "MOM": 0.70, "RMW": 0.30, "CMA": 0.25, "idio_vol_ann": 0.055},
        {"portfolio": "True_Alpha", "alpha_ann": 0.030, "MKT": 0.85, "SMB": 0.10, "HML": 0.10, "MOM": 0.20, "RMW": 0.25, "CMA": 0.10, "idio_vol_ann": 0.050},
        {"portfolio": "Noisy_Manager", "alpha_ann": 0.000, "MKT": 0.80, "SMB": 0.10, "HML": 0.00, "MOM": 0.00, "RMW": 0.00, "CMA": 0.00, "idio_vol_ann": 0.150},
    ]

    true_exposures = pd.DataFrame(exposure_rows)

    factor_cols = ["MKT", "SMB", "HML", "MOM", "RMW", "CMA"]
    factor_matrix = factors[factor_cols].to_numpy()

    rows = []

    for _, row in true_exposures.iterrows():
        beta = row[factor_cols].to_numpy(dtype=float)
        alpha_daily = row["alpha_ann"] / config.annualisation
        idio_vol_daily = row["idio_vol_ann"] / np.sqrt(config.annualisation)

        residual = idio_vol_daily * rng.standard_t(df=7, size=len(factors)) * np.sqrt((7 - 2) / 7)
        excess_return = alpha_daily + factor_matrix @ beta + residual
        total_return = factors["RF"].to_numpy() + excess_return

        for date, r_total, r_excess in zip(factors["date"], total_return, excess_return):
            rows.append({
                "date": date,
                "portfolio": row["portfolio"],
                "portfolio_return": r_total,
                "portfolio_excess_return": r_excess
            })

    portfolio_panel = pd.DataFrame(rows)

    return portfolio_panel, true_exposures


portfolio_panel, true_exposures = simulate_portfolio_returns(factors, config)

portfolio_panel.head(), true_exposures

In [ ]:
portfolio_returns = portfolio_panel.pivot(index="date", columns="portfolio", values="portfolio_return").sort_index()
portfolio_excess_returns = portfolio_panel.pivot(index="date", columns="portfolio", values="portfolio_excess_return").sort_index()

plt.figure(figsize=(12, 6))
for col in portfolio_returns.columns:
    plt.plot(portfolio_returns.index, (1 + portfolio_returns[col]).cumprod(), label=col, alpha=0.85)
plt.title("Synthetic Portfolio Cumulative Returns")
plt.xlabel("Date")
plt.ylabel("Growth of 1")
plt.legend(ncol=2)
plt.show()

## 7. Basic performance metrics

Before factor attribution, compute basic performance:

- annualised return;
- annualised volatility;
- Sharpe ratio proxy;
- max drawdown;
- total return.

This is the performance we want to explain.

In [ ]:
def max_drawdown_from_returns(r):
    nav = (1 + r).cumprod()
    drawdown = nav / nav.cummax() - 1
    return float(drawdown.min())


def performance_metrics(return_matrix, rf=None, annualisation=252):
    rows = []

    for col in return_matrix.columns:
        r = return_matrix[col].dropna()
        excess = r - rf.reindex(r.index) if rf is not None else r

        rows.append({
            "portfolio": col,
            "annualised_return": float(r.mean() * annualisation),
            "annualised_vol": float(r.std(ddof=1) * np.sqrt(annualisation)),
            "sharpe_proxy": float(excess.mean() / r.std(ddof=1) * np.sqrt(annualisation)) if r.std(ddof=1) > 0 else np.nan,
            "max_drawdown": max_drawdown_from_returns(r),
            "total_return": float((1 + r).prod() - 1),
        })

    return pd.DataFrame(rows).sort_values("sharpe_proxy", ascending=False)


rf_series = factors.set_index("date")["RF"]
basic_performance = performance_metrics(portfolio_returns, rf=rf_series, annualisation=config.annualisation)

basic_performance

## 8. Factor model definitions

We compare:

```text
CAPM        = MKT
FF3         = MKT + SMB + HML
Carhart4    = MKT + SMB + HML + MOM
FF5         = MKT + SMB + HML + RMW + CMA
FF5_MOM     = MKT + SMB + HML + RMW + CMA + MOM
```

The last model is not a separate historical named model here. It is a practical combined specification.

In [ ]:
factor_models = {
    "CAPM": ["MKT"],
    "FF3": ["MKT", "SMB", "HML"],
    "Carhart4": ["MKT", "SMB", "HML", "MOM"],
    "FF5": ["MKT", "SMB", "HML", "RMW", "CMA"],
    "FF5_MOM": ["MKT", "SMB", "HML", "RMW", "CMA", "MOM"],
}

factor_models

## 9. OLS factor regression

For each portfolio and model:

$$
R_p-R_f=\alpha+\beta^\top f+\epsilon
$$

We estimate:

- alpha;
- betas;
- residual volatility;
- $R^2$;
- t-statistics;
- annualised alpha.

In [ ]:
def fit_ols_factor_model(y, X, factor_names):
    y = np.asarray(y, dtype=float)
    X = np.asarray(X, dtype=float)

    X_design = np.column_stack([np.ones(len(X)), X])
    beta = np.linalg.pinv(X_design.T @ X_design) @ X_design.T @ y

    fitted = X_design @ beta
    residual = y - fitted

    n, k = X_design.shape
    dof = max(n - k, 1)
    sigma2 = float(residual @ residual / dof)

    xtx_inv = np.linalg.pinv(X_design.T @ X_design)
    se = np.sqrt(np.diag(sigma2 * xtx_inv))
    t_stats = beta / se

    ssr = float(residual @ residual)
    sst = float(((y - y.mean()) @ (y - y.mean())))
    r2 = 1 - ssr / sst if sst > 0 else np.nan

    result = {
        "intercept": float(beta[0]),
        "alpha_daily": float(beta[0]),
        "alpha_ann": float(beta[0] * config.annualisation),
        "resid_vol_ann": float(np.std(residual, ddof=1) * np.sqrt(config.annualisation)),
        "r2": float(r2),
        "n_obs": int(n),
        "sigma2": sigma2,
        "residual": residual,
        "fitted": fitted,
        "coef_vector": beta,
        "se_vector": se,
        "t_vector": t_stats,
    }

    for i, name in enumerate(factor_names, start=1):
        result[f"beta_{name}"] = float(beta[i])
        result[f"t_beta_{name}"] = float(t_stats[i])

    result["t_alpha"] = float(t_stats[0])

    return result


def run_factor_regressions(excess_returns, factors, factor_models):
    factor_df = factors.set_index("date")
    rows = []

    for portfolio in excess_returns.columns:
        y_full = excess_returns[portfolio].dropna()

        for model_name, model_factors in factor_models.items():
            joined = pd.concat([y_full.rename("y"), factor_df[model_factors]], axis=1).dropna()
            fit = fit_ols_factor_model(joined["y"], joined[model_factors], model_factors)

            row = {
                "portfolio": portfolio,
                "model": model_name,
                "factors": "|".join(model_factors),
                "alpha_ann": fit["alpha_ann"],
                "t_alpha": fit["t_alpha"],
                "resid_vol_ann": fit["resid_vol_ann"],
                "r2": fit["r2"],
                "n_obs": fit["n_obs"],
            }

            for key, value in fit.items():
                if key.startswith("beta_") or key.startswith("t_beta_"):
                    row[key] = value

            rows.append(row)

    return pd.DataFrame(rows)


regression_results_full = run_factor_regressions(portfolio_excess_returns, factors, factor_models)

regression_results_full.head()

## 10. Alpha shrinkage across models

A core attribution lesson:

> If CAPM alpha disappears under FF3, Carhart, or FF5, it was probably not pure alpha. It was exposure to omitted systematic factors.

We compare annualised alpha by model.

In [ ]:
alpha_pivot = regression_results_full.pivot(index="portfolio", columns="model", values="alpha_ann")
alpha_pivot = alpha_pivot[["CAPM", "FF3", "Carhart4", "FF5", "FF5_MOM"]]

alpha_pivot

In [ ]:
plt.figure(figsize=(12, 6))
for model in alpha_pivot.columns:
    plt.plot(alpha_pivot.index, alpha_pivot[model], marker="o", label=model)
plt.axhline(0, linestyle="--")
plt.xticks(rotation=45, ha="right")
plt.title("Annualised Alpha by Factor Model")
plt.xlabel("Portfolio")
plt.ylabel("Annualised alpha")
plt.legend()
plt.tight_layout()
plt.show()

## 11. CAPM alpha versus richer-model alpha

This chart focuses on the key lesson.

A portfolio such as `Fake_CAPM_Alpha` can look impressive under CAPM, then lose alpha once size, value, momentum, profitability, and investment exposures are included.

In [ ]:
alpha_comparison = alpha_pivot[["CAPM", "FF5_MOM"]].copy()
alpha_comparison["alpha_shrinkage_CAPM_to_FF5_MOM"] = alpha_comparison["FF5_MOM"] - alpha_comparison["CAPM"]

alpha_comparison.sort_values("alpha_shrinkage_CAPM_to_FF5_MOM")

In [ ]:
plt.figure(figsize=(10, 6))
x = np.arange(len(alpha_comparison.index))
width = 0.35
plt.bar(x - width/2, alpha_comparison["CAPM"], width=width, label="CAPM alpha")
plt.bar(x + width/2, alpha_comparison["FF5_MOM"], width=width, label="FF5+MOM alpha")
plt.axhline(0, linestyle="--")
plt.xticks(x, alpha_comparison.index, rotation=45, ha="right")
plt.title("CAPM Alpha vs FF5+Momentum Alpha")
plt.ylabel("Annualised alpha")
plt.legend()
plt.tight_layout()
plt.show()

## 12. Train/test attribution

To avoid overfitting attribution, estimate factor exposures on the training period and apply them to the test period.

This mimics live performance attribution:

1. estimate betas using historical data;
2. observe new returns;
3. explain new returns with old betas;
4. track residual and alpha.

In [ ]:
unique_dates = portfolio_excess_returns.index
split_idx = int(len(unique_dates) * config.train_fraction)
train_dates = unique_dates[:split_idx]
test_dates = unique_dates[split_idx:]

factors_wide = factors.set_index("date")

train_excess = portfolio_excess_returns.loc[train_dates]
test_excess = portfolio_excess_returns.loc[test_dates]
train_factors = factors_wide.loc[train_dates]
test_factors = factors_wide.loc[test_dates]

pd.Series({
    "n_train_days": len(train_dates),
    "n_test_days": len(test_dates),
    "train_start": str(train_dates[0]),
    "train_end": str(train_dates[-1]),
    "test_start": str(test_dates[0]),
    "test_end": str(test_dates[-1])
})

In [ ]:
def fit_models_on_train(train_excess, train_factors, factor_models):
    fitted = {}

    for portfolio in train_excess.columns:
        fitted[portfolio] = {}

        for model_name, model_factors in factor_models.items():
            joined = pd.concat([
                train_excess[portfolio].rename("y"),
                train_factors[model_factors]
            ], axis=1).dropna()

            fit = fit_ols_factor_model(joined["y"], joined[model_factors], model_factors)
            fitted[portfolio][model_name] = fit

    return fitted


train_fits = fit_models_on_train(train_excess, train_factors, factor_models)

list(train_fits.keys())[:3], list(train_fits["Fake_CAPM_Alpha"].keys())

## 13. Out-of-sample attribution table

Using train-estimated betas, compute test-period contribution:

$$
\begin{aligned}
\widehat R_{p,t}^{excess} &= \hat \alpha \\
&\quad + \sum_j \hat\beta_j f_{j,t}
\end{aligned}
$$

Realised residual:

$$
\begin{aligned}
\hat\epsilon_t &= R_{p,t}^{excess} \\
&\quad - \widehat R_{p,t}^{excess}
\end{aligned}
$$

Annualised contribution:

$$
252 \times mean(contribution_t)
$$

In [ ]:
def test_period_attribution(test_excess, test_factors, train_fits, factor_models):
    rows = []
    daily_contrib_frames = []

    for portfolio in test_excess.columns:
        y = test_excess[portfolio].dropna()

        for model_name, model_factors in factor_models.items():
            fit = train_fits[portfolio][model_name]
            alpha_daily = fit["alpha_daily"]
            beta_vec = np.array([fit[f"beta_{f}"] for f in model_factors])

            X = test_factors.loc[y.index, model_factors]
            factor_contrib = X.multiply(beta_vec, axis=1)
            alpha_contrib = pd.Series(alpha_daily, index=y.index, name="alpha")
            fitted = alpha_contrib + factor_contrib.sum(axis=1)
            residual = y - fitted

            row = {
                "portfolio": portfolio,
                "model": model_name,
                "realised_excess_ann": float(y.mean() * config.annualisation),
                "predicted_excess_ann": float(fitted.mean() * config.annualisation),
                "alpha_contribution_ann": float(alpha_contrib.mean() * config.annualisation),
                "residual_contribution_ann": float(residual.mean() * config.annualisation),
                "tracking_error_ann": float(residual.std(ddof=1) * np.sqrt(config.annualisation)),
                "test_r2_proxy": float(1 - (residual @ residual) / ((y - y.mean()) @ (y - y.mean()))) if ((y - y.mean()) @ (y - y.mean())) > 0 else np.nan,
            }

            for f in model_factors:
                row[f"{f}_contribution_ann"] = float(factor_contrib[f].mean() * config.annualisation)

            rows.append(row)

            daily = factor_contrib.copy()
            daily["alpha"] = alpha_contrib
            daily["residual"] = residual
            daily["portfolio"] = portfolio
            daily["model"] = model_name
            daily["date"] = y.index
            daily_contrib_frames.append(daily.reset_index(drop=True))

    return pd.DataFrame(rows), pd.concat(daily_contrib_frames, ignore_index=True)


test_attr, daily_attribution = test_period_attribution(test_excess, test_factors, train_fits, factor_models)

test_attr.head()

In [ ]:
test_attr[
    (test_attr["portfolio"].isin(["Fake_CAPM_Alpha", "True_Alpha", "Momentum_Growth"]))
    & (test_attr["model"].isin(["CAPM", "FF5_MOM"]))
].sort_values(["portfolio", "model"])

## 14. Attribution waterfall for one portfolio

We build an attribution waterfall for:

```text
Fake_CAPM_Alpha
```

under:

```text
CAPM
FF5_MOM
```

The expectation is that CAPM attributes too much to alpha, while the richer model attributes more to systematic factor exposures.

In [ ]:
def attribution_bar_data(test_attr, portfolio, model):
    row = test_attr[(test_attr["portfolio"] == portfolio) & (test_attr["model"] == model)].iloc[0]

    contribution_cols = [c for c in test_attr.columns if c.endswith("_contribution_ann")]
    values = []

    for col in contribution_cols:
        val = row.get(col, np.nan)
        if pd.notna(val):
            label = col.replace("_contribution_ann", "")
            values.append({"component": label, "annualised_contribution": val})

    return pd.DataFrame(values)


portfolio_to_plot = "Fake_CAPM_Alpha"

for model in ["CAPM", "FF5_MOM"]:
    bar_df = attribution_bar_data(test_attr, portfolio_to_plot, model)

    plt.figure(figsize=(10, 6))
    plt.barh(bar_df["component"], bar_df["annualised_contribution"])
    plt.axvline(0, linestyle="--")
    plt.title(f"Test Attribution: {portfolio_to_plot}, {model}")
    plt.xlabel("Annualised contribution")
    plt.ylabel("Component")
    plt.show()

## 15. Factor exposures versus truth

Because this is synthetic, we know the true betas.

We compare estimated FF5+MOM exposures against true exposures.

In [ ]:
def estimated_beta_table(regression_results, model="FF5_MOM"):
    rows = []
    sub = regression_results[regression_results["model"] == model]

    for _, row in sub.iterrows():
        out = {"portfolio": row["portfolio"]}
        for factor in factor_models[model]:
            out[f"estimated_{factor}"] = row.get(f"beta_{factor}", np.nan)
        rows.append(out)

    return pd.DataFrame(rows)


estimated_betas = estimated_beta_table(regression_results_full, model="FF5_MOM")

true_beta_cols = ["portfolio", "MKT", "SMB", "HML", "MOM", "RMW", "CMA"]
truth_compare = true_exposures[true_beta_cols].merge(estimated_betas, on="portfolio", how="left")

truth_compare.head()

In [ ]:
beta_error_rows = []

for factor in ["MKT", "SMB", "HML", "MOM", "RMW", "CMA"]:
    err = truth_compare[f"estimated_{factor}"] - truth_compare[factor]
    beta_error_rows.append({
        "factor": factor,
        "mean_abs_error": float(np.mean(np.abs(err))),
        "rmse": float(np.sqrt(np.mean(err**2))),
        "bias": float(np.mean(err))
    })

beta_error_report = pd.DataFrame(beta_error_rows)

beta_error_report

## 16. Systematic versus residual risk

Using train-estimated factor exposures:

$$
\begin{aligned}
Var(R_p) &= \beta^\top \Sigma_f \beta \\
&\quad + \sigma_\epsilon^2
\end{aligned}
$$

Systematic risk share:

$$
\frac{\beta^\top \Sigma_f \beta}{Var(R_p)}
$$

This helps separate factor-driven risk from idiosyncratic manager risk.

In [ ]:
def risk_decomposition(train_fits, train_factors, factor_models):
    rows = []

    for portfolio, model_dict in train_fits.items():
        for model_name, fit in model_dict.items():
            model_factors = factor_models[model_name]
            beta = np.array([fit[f"beta_{f}"] for f in model_factors])
            cov_f = train_factors[model_factors].cov().to_numpy()

            systematic_var_daily = float(beta.T @ cov_f @ beta)
            residual_var_daily = float((fit["resid_vol_ann"] / np.sqrt(config.annualisation)) ** 2)
            total_var_daily = systematic_var_daily + residual_var_daily

            rows.append({
                "portfolio": portfolio,
                "model": model_name,
                "systematic_vol_ann": float(np.sqrt(max(systematic_var_daily, 0)) * np.sqrt(config.annualisation)),
                "residual_vol_ann": float(np.sqrt(max(residual_var_daily, 0)) * np.sqrt(config.annualisation)),
                "total_model_vol_ann": float(np.sqrt(max(total_var_daily, 0)) * np.sqrt(config.annualisation)),
                "systematic_risk_share": float(systematic_var_daily / total_var_daily) if total_var_daily > 0 else np.nan,
                "residual_risk_share": float(residual_var_daily / total_var_daily) if total_var_daily > 0 else np.nan,
            })

    return pd.DataFrame(rows)


risk_decomp = risk_decomposition(train_fits, train_factors, factor_models)

risk_decomp.head()

In [ ]:
risk_plot = risk_decomp[risk_decomp["model"] == "FF5_MOM"].sort_values("residual_risk_share")

plt.figure(figsize=(10, 6))
plt.barh(risk_plot["portfolio"], risk_plot["systematic_risk_share"], label="systematic")
plt.barh(risk_plot["portfolio"], risk_plot["residual_risk_share"], left=risk_plot["systematic_risk_share"], label="residual")
plt.title("Risk Decomposition, FF5+MOM")
plt.xlabel("Risk share")
plt.ylabel("Portfolio")
plt.legend()
plt.show()

## 17. Rolling CAPM beta and alpha

Static betas can hide changing exposures.

We compute rolling CAPM beta and alpha:

$$
R_p-R_f=\alpha_t+\beta_t MKT+\epsilon_t
$$

over a trailing window.

In [ ]:
def rolling_capm(portfolio_excess_returns, factors, window):
    factor_df = factors.set_index("date")
    rows = []

    for portfolio in portfolio_excess_returns.columns:
        joined = pd.concat([
            portfolio_excess_returns[portfolio].rename("y"),
            factor_df["MKT"]
        ], axis=1).dropna()

        for end_idx in range(window, len(joined) + 1):
            sub = joined.iloc[end_idx - window:end_idx]
            fit = fit_ols_factor_model(sub["y"], sub[["MKT"]], ["MKT"])

            rows.append({
                "date": sub.index[-1],
                "portfolio": portfolio,
                "rolling_alpha_ann": fit["alpha_ann"],
                "rolling_beta_MKT": fit["beta_MKT"],
                "rolling_r2": fit["r2"]
            })

    return pd.DataFrame(rows)


rolling_capm_results = rolling_capm(portfolio_excess_returns, factors, config.rolling_window)

rolling_capm_results.head()

In [ ]:
selected_portfolios = ["Market_Core", "Fake_CAPM_Alpha", "True_Alpha", "Noisy_Manager"]

plt.figure(figsize=(12, 6))
for portfolio in selected_portfolios:
    sub = rolling_capm_results[rolling_capm_results["portfolio"] == portfolio]
    plt.plot(sub["date"], sub["rolling_beta_MKT"], label=portfolio)
plt.title("Rolling CAPM Market Beta")
plt.xlabel("Date")
plt.ylabel("Rolling beta")
plt.legend()
plt.show()

plt.figure(figsize=(12, 6))
for portfolio in selected_portfolios:
    sub = rolling_capm_results[rolling_capm_results["portfolio"] == portfolio]
    plt.plot(sub["date"], sub["rolling_alpha_ann"], label=portfolio)
plt.axhline(0, linestyle="--")
plt.title("Rolling CAPM Annualised Alpha")
plt.xlabel("Date")
plt.ylabel("Rolling alpha")
plt.legend()
plt.show()

## 18. Benchmark-relative attribution

Suppose `Market_Core` is the benchmark.

Active return:

$$
R^{active}_{p,t} = R_{p,t}-R_{benchmark,t}
$$

Active attribution asks:

> Did the manager beat the benchmark because of factor tilts or residual alpha?

We run factor regressions on active returns.

In [ ]:
benchmark = "Market_Core"

active_returns = portfolio_returns.subtract(portfolio_returns[benchmark], axis=0)
active_excess_returns = active_returns.copy()

active_regression_results = run_factor_regressions(active_excess_returns.drop(columns=[benchmark]), factors, factor_models)

active_regression_results.head()

In [ ]:
active_alpha = active_regression_results.pivot(index="portfolio", columns="model", values="alpha_ann")
active_alpha = active_alpha[["CAPM", "FF3", "Carhart4", "FF5", "FF5_MOM"]]

active_alpha

## 19. Stress-day attribution

Factor models are averages.

During stress days, factor exposures can dominate performance.

We compare average portfolio excess returns on stress versus normal days.

In [ ]:
stress_series = factors.set_index("date")["stress_day"].reindex(portfolio_excess_returns.index)

stress_performance = []

for portfolio in portfolio_excess_returns.columns:
    r = portfolio_excess_returns[portfolio]

    stress_performance.append({
        "portfolio": portfolio,
        "mean_excess_return_normal_ann": float(r[~stress_series].mean() * config.annualisation),
        "mean_excess_return_stress_ann": float(r[stress_series].mean() * config.annualisation),
        "worst_stress_day_return": float(r[stress_series].min()),
        "n_stress_days": int(stress_series.sum())
    })

stress_performance = pd.DataFrame(stress_performance).sort_values("mean_excess_return_stress_ann")

stress_performance

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(stress_performance["portfolio"], stress_performance["mean_excess_return_stress_ann"])
plt.axvline(0, linestyle="--")
plt.title("Annualised Mean Excess Return on Stress Days")
plt.xlabel("Annualised stress-day excess return")
plt.ylabel("Portfolio")
plt.show()

## 20. Model comparison summary

We compare models by:

1. mean absolute alpha;
2. mean residual volatility;
3. mean $R^2$;
4. number of factors.

A richer model should explain more return variation, but too many factors can overfit or reduce interpretability.

In [ ]:
model_comparison = (
    regression_results_full
    .groupby("model")
    .agg(
        n_portfolios=("portfolio", "nunique"),
        mean_abs_alpha_ann=("alpha_ann", lambda x: np.mean(np.abs(x))),
        mean_resid_vol_ann=("resid_vol_ann", "mean"),
        mean_r2=("r2", "mean")
    )
    .reset_index()
)

model_comparison["n_factors"] = model_comparison["model"].map({k: len(v) for k, v in factor_models.items()})
model_comparison = model_comparison.sort_values("n_factors")

model_comparison

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(model_comparison["n_factors"], model_comparison["mean_abs_alpha_ann"], marker="o", label="mean abs alpha")
plt.plot(model_comparison["n_factors"], model_comparison["mean_resid_vol_ann"], marker="x", label="mean residual vol")
plt.title("Model Richness vs Unexplained Performance")
plt.xlabel("Number of factors")
plt.ylabel("Annualised quantity")
plt.legend()
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(model_comparison["n_factors"], model_comparison["mean_r2"], marker="o")
plt.title("Model Richness vs Mean R²")
plt.xlabel("Number of factors")
plt.ylabel("Mean R²")
plt.show()

## 21. Practical interpretation guide

### If CAPM alpha is positive but FF5+MOM alpha is near zero

Likely explanation:

> The manager had systematic tilts that CAPM omitted.

### If CAPM alpha and FF5+MOM alpha are both positive

Possible explanation:

> There may be true alpha, omitted factors, or noise.

### If residual volatility is high

Possible explanation:

> Portfolio has idiosyncratic bets, unstable exposures, or missing factors.

### If rolling beta changes

Possible explanation:

> Strategy exposure is time-varying and static attribution is incomplete.

### If stress-day performance is bad

Possible explanation:

> Factor exposures hide downside convexity or crash risk.

## 22. Governance checklist for performance attribution

Before claiming alpha:

1. **Start with CAPM**  
   Estimate market beta and first-pass alpha.

2. **Add richer factor models**  
   Check FF3, Carhart4, FF5, and FF5+momentum.

3. **Use train/test attribution**  
   Estimate betas historically and attribute later returns.

4. **Check alpha shrinkage**  
   Does alpha disappear after adding factors?

5. **Check t-statistics**  
   Is alpha statistically distinguishable from noise?

6. **Check residual risk**  
   Is unexplained volatility too high?

7. **Check rolling exposures**  
   Are betas stable?

8. **Check stress days**  
   Does performance collapse in factor stress?

9. **Check benchmark-relative returns**  
   Did the manager beat the correct benchmark?

10. **Avoid model worship**  
   Factor models are lenses, not truth.

## 23. Saving outputs

The notebook saves:

1. factor returns;
2. synthetic portfolio returns;
3. true exposures;
4. basic performance;
5. full regression results;
6. alpha shrinkage table;
7. train/test attribution;
8. daily attribution;
9. beta error report;
10. risk decomposition;
11. rolling CAPM results;
12. active attribution;
13. stress-day report;
14. model comparison;
15. manifest.

In [ ]:
output_dir = Path("data/processed/capm_and_performance_attribution_v1")
output_dir.mkdir(parents=True, exist_ok=True)

factors_path = output_dir / "factor_returns.csv"
portfolio_panel_path = output_dir / "portfolio_return_panel.csv"
portfolio_returns_path = output_dir / "portfolio_returns_wide.csv"
portfolio_excess_path = output_dir / "portfolio_excess_returns_wide.csv"
true_exposures_path = output_dir / "true_exposures.csv"
basic_performance_path = output_dir / "basic_performance.csv"
regression_results_path = output_dir / "factor_regression_results_full.csv"
alpha_pivot_path = output_dir / "alpha_by_model.csv"
alpha_comparison_path = output_dir / "alpha_comparison_capm_vs_ff5_mom.csv"
test_attr_path = output_dir / "test_period_attribution.csv"
daily_attr_path = output_dir / "daily_attribution.csv"
truth_compare_path = output_dir / "estimated_vs_true_betas.csv"
beta_error_path = output_dir / "beta_error_report.csv"
risk_decomp_path = output_dir / "risk_decomposition.csv"
rolling_capm_path = output_dir / "rolling_capm.csv"
active_regression_path = output_dir / "active_regression_results.csv"
active_alpha_path = output_dir / "active_alpha_by_model.csv"
stress_perf_path = output_dir / "stress_day_performance.csv"
model_comparison_path = output_dir / "model_comparison.csv"
manifest_path = output_dir / "manifest.json"

factors.to_csv(factors_path, index=False)
portfolio_panel.to_csv(portfolio_panel_path, index=False)
portfolio_returns.to_csv(portfolio_returns_path)
portfolio_excess_returns.to_csv(portfolio_excess_path)
true_exposures.to_csv(true_exposures_path, index=False)
basic_performance.to_csv(basic_performance_path, index=False)
regression_results_full.to_csv(regression_results_path, index=False)
alpha_pivot.to_csv(alpha_pivot_path)
alpha_comparison.to_csv(alpha_comparison_path)
test_attr.to_csv(test_attr_path, index=False)
daily_attribution.to_csv(daily_attr_path, index=False)
truth_compare.to_csv(truth_compare_path, index=False)
beta_error_report.to_csv(beta_error_path, index=False)
risk_decomp.to_csv(risk_decomp_path, index=False)
rolling_capm_results.to_csv(rolling_capm_path, index=False)
active_regression_results.to_csv(active_regression_path, index=False)
active_alpha.to_csv(active_alpha_path)
stress_performance.to_csv(stress_perf_path, index=False)
model_comparison.to_csv(model_comparison_path, index=False)

manifest = {
    "dataset_name": "capm_and_performance_attribution_outputs",
    "schema_version": "capm_and_performance_attribution_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "factor_models": factor_models,
    "portfolios": portfolio_returns.columns.tolist(),
    "train_test_split": {
        "n_train_days": int(len(train_dates)),
        "n_test_days": int(len(test_dates)),
        "train_start": str(train_dates[0]),
        "train_end": str(train_dates[-1]),
        "test_start": str(test_dates[0]),
        "test_end": str(test_dates[-1])
    },
    "model_comparison": model_comparison.to_dict(orient="records"),
    "notes": [
        "CAPM is treated as a baseline attribution model, not the final truth.",
        "Synthetic portfolios include known factor exposures and one true-alpha portfolio.",
        "Fake_CAPM_Alpha is designed to show how CAPM can overstate alpha when factors are omitted.",
        "Train/test attribution estimates betas on the train period and applies them to the test period.",
        "FF5_MOM is used as a practical combined specification including Fama-French 5 factors plus momentum.",
        "This notebook is educational and synthetic; real attribution requires point-in-time factor data and careful benchmark selection."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

factors_path, regression_results_path, test_attr_path, model_comparison_path, manifest_path

## 24. Limitations

### 24.1 Synthetic data

The notebook uses synthetic factor returns and portfolio returns.

Real performance attribution needs actual factor data, correct benchmark returns, fees, cash flows, and survivorship controls.

### 24.2 Factor models are incomplete

CAPM, FF3, Carhart4, and FF5 are useful lenses, but not complete descriptions of expected returns.

Missing factors can still appear as alpha.

### 24.3 Linear exposure assumption

OLS assumes constant linear betas.

Real portfolios may have nonlinear exposures, options, timing, leverage, and regime-dependent risks.

### 24.4 Statistical uncertainty

Alpha t-stats can be misleading when residuals are autocorrelated, heteroskedastic, or non-normal.

Production analysis should consider robust standard errors.

### 24.5 Synthetic true alpha

The `True_Alpha` portfolio has built-in alpha by construction.

In real life, true alpha is never directly observable.

### 24.6 Benchmark selection

Active attribution depends heavily on the benchmark.

The wrong benchmark can create fake alpha.

### 24.7 Factor definitions vary

Different vendors and regions define factors differently.

Attribution results can change with factor construction.

## 25. Research frontier and extensions

Important extensions include:

1. **Robust standard errors**  
   Newey-West alpha t-statistics.

2. **Rolling multi-factor exposures**  
   Time-varying betas for all factors.

3. **Bayesian factor regression**  
   Shrink noisy beta estimates.

4. **Nonlinear exposure models**  
   Capture option-like payoffs and convexity.

5. **Holdings-based attribution**  
   Decompose returns using actual positions.

6. **Brinson attribution**  
   Allocation, selection, and interaction effects.

7. **Risk-based attribution**  
   Decompose volatility and drawdown by factor.

8. **Regime-dependent attribution**  
   Different betas in stress versus calm periods.

9. **International factor attribution**  
   Add currency, country, and sector factors.

10. **Futures strategy attribution**  
   Trend, carry, roll yield, curve, and volatility factors for commodity futures.

## 26. Suggested follow-up notebooks

This notebook naturally leads to:

1. **04_03_covariance_estimation_and_shrinkage**  
   Estimate risk inputs from return matrices.

2. **04_04_risk_model_factor_exposures**  
   Build explicit portfolio factor risk models.

3. **04_05_mean_variance_optimization_constraints**  
   Use expected returns and covariance for allocation.

4. **04_06_risk_parity_and_volatility_targeting**  
   Allocate by risk contribution.

5. **04_07_var_cvar_and_stress_testing**  
   Tail risk and drawdown analysis.

6. **05_02_performance_attribution_report_template**  
   Turn this attribution into a reusable reporting workflow.

## 27. Summary

This notebook implemented CAPM and multi-factor performance attribution.

It showed how to:

1. simulate factor returns;
2. simulate portfolios with known exposures;
3. compute basic performance;
4. estimate CAPM alpha and beta;
5. estimate FF3, Carhart4, FF5, and FF5+MOM models;
6. compare alpha shrinkage across models;
7. perform train/test attribution;
8. decompose returns into alpha, factor contributions, and residuals;
9. compare estimated betas with known synthetic truth;
10. decompose systematic and residual risk;
11. compute rolling CAPM beta and alpha;
12. run benchmark-relative attribution;
13. inspect stress-day performance;
14. save outputs and metadata.

The key computational takeaway:

> Performance attribution is a regression and accounting problem: define factors, estimate exposures, decompose returns, and audit residuals.

The key financial takeaway:

> Alpha is what remains after the right risks have been accounted for, not what remains after the easiest model.

## 28. Further reading

- Sharpe, Lintner, and Mossin on CAPM.
- Fama and French on 3-factor and 5-factor models.
- Carhart on momentum and mutual fund performance persistence.
- Grinold and Kahn, *Active Portfolio Management.*
- Factor investing, performance attribution, and risk model literature.